# 1.3 Dataset split v2 — court-grouped

Copy of `1.2_embeddings_encodings.ipynb`, reading `legal_paragraphs_clean.parquet` and
writing `legal_paragraphs_dataset_v2.parquet`. The 1.2 notebook and its output are frozen.

The v1 split groups by `document_id`, which prevents paragraphs of one judgment from
straddling train and test but does nothing about the court. Because documents are clustered
by issuing court (ICC ~= 0.17 on log paragraph count), **42.7% of v1 test documents come from
a court that also appears in train** — the model can score well by memorising a court's
template. This notebook keeps the v1 split for comparability and adds a second, court-grouped
split so the two can be trained side by side.

In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
CLEAN_PATH = "legal_paragraphs_clean.parquet"
FROZEN_DATASET_PATH = "legal_paragraphs_dataset.parquet"  # read-only, for split inheritance
OUTPUT_PATH = "legal_paragraphs_dataset_v2.parquet"

N_SPLITS = 10
RANDOM_STATE = 42
TEST_FOLD, VALIDATION_FOLD = 0, 1

QUALITY_FLAGS = ["section_scrambled", "section_missing", "extreme_length"]

In [3]:
p_df = pd.read_parquet(CLEAN_PATH)
p_df["document_id"] = p_df["document_id"].astype(str)
p_df = p_df.sort_values(["document_id", "paragraph_index"], kind="stable").reset_index(drop=True)
p_df.shape, p_df.columns.tolist()

((281725, 10),
 ['document_id',
  'paragraph_index',
  'paragraph_order',
  'numbered_text',
  'text',
  'section',
  'section_scrambled',
  'section_missing',
  'extreme_length',
  'n_paragraphs'])

## Labels

Pinned to the mapping 1.2 produced, rather than re-derived from `unique()` — the ordering of
`unique()` depends on row order, and this notebook sorts before it would have been called.
The assert ties the constant back to the frozen dataset.

In [4]:
LABELS_TO_IDS = {"introductory": 0, "reasoning": 1, "operative": 2}
IDS_TO_LABELS = {value: key for key, value in LABELS_TO_IDS.items()}

frozen_df = pd.read_parquet(FROZEN_DATASET_PATH, columns=["document_id", "paragraph_index", "section", "label", "split"])
frozen_df["document_id"] = frozen_df["document_id"].astype(str)
assert frozen_df.drop_duplicates(["section", "label"]).set_index("section")["label"].to_dict() == LABELS_TO_IDS

assert set(p_df["section"].dropna()).issubset(LABELS_TO_IDS)
LABELS_TO_IDS

{'introductory': 0, 'reasoning': 1, 'operative': 2}

## Context (unchanged from 1.2)

In [5]:
by_document = p_df.groupby("document_id")["text"]
previous_1 = by_document.shift(1).fillna("")
previous_2 = by_document.shift(2).fillna("")

p_df["context"] = (previous_2 + "\n\n" + previous_1).str.strip()
p_df["label"] = p_df["section"].map(LABELS_TO_IDS).astype("int8")

assert not p_df.duplicated(["document_id", "paragraph_index"]).any()
p_df.head()

,document_id,paragraph_index,paragraph_order,numbered_text,text,section,section_scrambled,section_missing,extreme_length,n_paragraphs,context,label
0,116061850,1,1,[paragraph_id=1] Справа № 127/30543/23,Справа № 127/30543/23,introductory,False,False,False,104,,0
1,116061850,2,2,[paragraph_id=2] Провадження № 1-кп/127/904/23,Провадження № 1-кп/127/904/23,introductory,False,False,False,104,Справа № 127/30543/23,0
2,116061850,3,3,[paragraph_id=3] В И Р О К,В И Р О К,introductory,False,False,False,104,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,0
3,116061850,4,4,[paragraph_id=4] ІМЕНЕМ УКРАЇНИ,ІМЕНЕМ УКРАЇНИ,introductory,False,False,False,104,Провадження № 1-кп/127/904/23\n\nВ И Р О К,0
4,116061850,5,5,[paragraph_id=5] 28.12.2023 |місто Вінниця|,28.12.2023 |місто Вінниця|,introductory,False,False,False,104,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,0


## Court extraction

There is no court column in the corpus, so it is recovered from the introductory section.
The pattern demands both a court-type word and a standalone `СУД` token, and rejects the
phrases that otherwise dominate the matches (`секретар судового засідання`, `головуючий у
суді`, document titles beginning `про призначення ... судового ...`).

This is deliberately high-precision: it matches 39% of documents with 859 distinct courts and
no junk in the top matches. A looser substring rule reaches 57% coverage but pulls in
document titles, which would merge unrelated courts into one group and *weaken* the split.
Unmatched documents become singleton groups keyed on their own id, so grouping degrades
gracefully to the v1 document-level behaviour for the part of the corpus we cannot attribute.

In [6]:
COURT_TYPE = (
    r"(РАЙОННИЙ|АПЕЛЯЦІЙНИЙ|МІСЬКИЙ|МІЖРАЙОННИЙ|ОКРУЖНИЙ|ГОСПОДАРСЬК\w+"
    r"|ВЕРХОВНИЙ|АНТИКОРУПЦІЙНИЙ|ВІЙСЬКОВИЙ|КАСАЦІЙНИЙ|РАЙСУД|АДМІНІСТРАТИВНИЙ)"
)
COURT_TOKEN = r"\bСУД(У|І|ОМ)?\b"
NOT_A_COURT = r"СУДД|ЗАСІДАН|ГОЛОВУЮЧ|СЕКРЕТАР|ПРИЗНАЧЕН"
MAX_COURT_NAME_LENGTH = 90

def extract_court(texts):
    """First paragraph that looks like a court name, normalised; None if nothing matches."""
    for text in texts:
        candidate = " ".join(text.split()).upper().strip(" ,.:;")
        if len(candidate) > MAX_COURT_NAME_LENGTH:
            continue
        if (re.search(COURT_TYPE, candidate)
                and re.search(COURT_TOKEN, candidate)
                and not re.search(NOT_A_COURT, candidate)):
            return candidate
    return None

introductory = p_df.loc[p_df["section"] == "introductory"]
court = introductory.groupby("document_id")["text"].apply(extract_court)
court = court.reindex(p_df["document_id"].unique())

coverage = court.notna().mean()
print(f"court matched for {coverage:.1%} of documents, {court.nunique()} distinct courts")
court.value_counts().head(10)

court matched for 39.1% of documents, 858 distinct courts


text
ПЕЧЕРСЬКИЙ РАЙОННИЙ СУД МІСТА КИЄВА                227
КИЇВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД                           82
ІНДУСТРІАЛЬНИЙ РАЙОННИЙ СУД М. ДНІПРОПЕТРОВСЬКА     77
ВІННИЦЬКИЙ АПЕЛЯЦІЙНИЙ СУД                          67
ЦЕНТРАЛЬНИЙ РАЙОННИЙ СУД М. МИКОЛАЄВА               57
КИЇВСЬКИЙ РАЙОННИЙ СУД М. ПОЛТАВИ                   52
ВИЩИЙ АНТИКОРУПЦІЙНИЙ СУД                           51
ЖИТОМИРСЬКИЙ АПЕЛЯЦІЙНИЙ СУД                        49
ДЗЕРЖИНСЬКИЙ РАЙОННИЙ СУД МІСТА КРИВОГО РОГУ        37
ДНІПРОВСЬКИЙ АПЕЛЯЦІЙНИЙ СУД                        35
Name: count, dtype: int64

In [7]:
# Unattributed documents get their own group so they never leak, but also never merge.
court_group = court.where(court.notna(), "DOC:" + court.index.to_series())
p_df["court"] = p_df["document_id"].map(court).astype("object")
p_df["court_group"] = p_df["document_id"].map(court_group)

group_sizes = court_group.value_counts()
print(f"groups: {len(group_sizes)} "
      f"({(group_sizes == 1).sum()} singletons, {(group_sizes >= 10).sum()} with >= 10 documents)")
print(f"documents in a group of >= 10: {group_sizes[group_sizes >= 10].sum() / len(court_group):.0%}")

groups: 6009 (5613 singletons, 70 with >= 10 documents)
documents in a group of >= 10: 21%


## Split A — `split`, inherited from v1

Not recomputed. `StratifiedGroupKFold` is sensitive to its input, so re-running it on the
cleaned frame would silently produce a *different* document-grouped split and the comparison
against the frozen `model_v2` run would no longer be clean. Inheriting the assignment makes
this arm byte-identical to v1 on every surviving document.

In [8]:
frozen_split = frozen_df.drop_duplicates("document_id").set_index("document_id")["split"]
missing = set(p_df["document_id"]) - set(frozen_split.index)
assert not missing, f"{len(missing)} cleaned documents absent from the frozen dataset"

p_df["split"] = p_df["document_id"].map(frozen_split)
assert p_df["split"].notna().all()

# Paragraph-level agreement with v1 wherever a document survived cleaning.
check = p_df.merge(frozen_df, on=["document_id", "paragraph_index"], suffixes=("", "_v1"))
assert len(check) == len(p_df)
assert (check["split"] == check["split_v1"]).all()
assert (check["label"] == check["label_v1"]).all()
del check

p_df["split"].value_counts()

split
train         224294
validation     28929
test           28502
Name: count, dtype: int64

## Split B — `split_court`, grouped by court

In [9]:
folds = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

p_df["court_fold"] = -1
for fold, (_, held_out_indicies) in enumerate(
    folds.split(p_df, y=p_df["label"], groups=p_df["court_group"])
):
    p_df.loc[held_out_indicies, "court_fold"] = fold

assert (p_df["court_fold"] >= 0).all()

p_df["split_court"] = "train"
p_df.loc[p_df["court_fold"] == TEST_FOLD, "split_court"] = "test"
p_df.loc[p_df["court_fold"] == VALIDATION_FOLD, "split_court"] = "validation"
p_df["split_court"].value_counts()

split_court
train         225386
test           28174
validation     28165
Name: count, dtype: int64

## Leakage check — the point of the exercise

In [10]:
def court_leakage(split_column):
    """Share of test documents whose court also appears in train. Singletons cannot leak."""
    documents = p_df.drop_duplicates("document_id")
    train_courts = set(documents.loc[documents[split_column] == "train", "court"].dropna())
    test_documents = documents.loc[documents[split_column] == "test"]
    attributed = test_documents["court"].notna()
    return {
        "n_test_documents": len(test_documents),
        "attributed_to_a_court": float(attributed.mean()),
        "leaked_share_of_all_test": float(test_documents["court"].isin(train_courts).mean()),
        "leaked_share_of_attributed": float(
            test_documents.loc[attributed, "court"].isin(train_courts).mean()
        ),
    }

leakage = pd.DataFrame({column: court_leakage(column) for column in ["split", "split_court"]})
leakage.round(4)

,split,split_court
n_test_documents,855.0000,839.0000
attributed_to_a_court,0.3918,0.3826
leaked_share_of_all_test,0.3275,0.0000
leaked_share_of_attributed,0.8358,0.0000


In [11]:
assert leakage.loc["leaked_share_of_all_test", "split_court"] == 0.0, "court leaked into split_court"
print("split_court: zero train/test court overlap")
print(f"split      : {leakage.loc['leaked_share_of_attributed', 'split']:.1%} of court-attributed "
      f"test documents share a court with train")

split_court: zero train/test court overlap
split      : 83.6% of court-attributed test documents share a court with train


## Stratification held?

In [12]:
for column in ["split", "split_court"]:
    print(f"--- {column} ---")
    print(pd.crosstab(p_df[column], p_df["section"], normalize="index").round(4).to_string())
    print(pd.crosstab(p_df[column], p_df["section"]).to_string())
    print()

--- split ---
section     introductory  operative  reasoning
split                                         
test              0.3024     0.2211     0.4764
train             0.3005     0.2311     0.4684
validation        0.2987     0.2273     0.4740
section     introductory  operative  reasoning
split                                         
test                8620       6303      13579
train              67400      51835     105059
validation          8642       6576      13711

--- split_court ---
section      introductory  operative  reasoning
split_court                                    
test               0.3006     0.2297     0.4697
train              0.3005     0.2297     0.4698
validation         0.3005     0.2297     0.4698
section      introductory  operative  reasoning
split_court                                    
test                 8468       6472      13234
train               67731      51773     105882
validation           8463       6469      13233



In [13]:
documents = p_df.drop_duplicates("document_id")
for column in ["split", "split_court"]:
    print(f"{column}: documents per split")
    print(documents[column].value_counts().to_string())
    print(documents.groupby(column)["n_paragraphs"].agg(["mean", "median", "max"]).round(1).to_string())
    print()

split: documents per split
split
train         6756
test           855
validation     849
            mean  median   max
split                         
test        33.3    27.0  1847
train       33.2    28.0   866
validation  34.1    27.0  1967

split_court: documents per split
split_court
train         6837
test           839
validation     784
             mean  median   max
split_court                    
test         33.6    28.0   257
train        33.0    28.0  1967
validation   35.9    29.0  1847



## Write

In [14]:
training_df = p_df[[
    "document_id",
    "paragraph_index",
    "context",
    "text",
    "section",
    "label",
    "court",
    "court_group",
    "split",
    "split_court",
    "n_paragraphs",
    *QUALITY_FLAGS,
]].copy()

training_df.to_parquet(OUTPUT_PATH, index=False)
training_df.shape, training_df.columns.tolist()

((281725, 14),
 ['document_id',
  'paragraph_index',
  'context',
  'text',
  'section',
  'label',
  'court',
  'court_group',
  'split',
  'split_court',
  'n_paragraphs',
  'section_scrambled',
  'section_missing',
  'extreme_length'])

In [15]:
training_df.head()

,document_id,paragraph_index,context,text,section,label,court,court_group,split,split_court,n_paragraphs,section_scrambled,section_missing,extreme_length
0,116061850,1,,Справа № 127/30543/23,introductory,0,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,train,train,104,False,False,False
1,116061850,2,Справа № 127/30543/23,Провадження № 1-кп/127/904/23,introductory,0,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,train,train,104,False,False,False
2,116061850,3,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,В И Р О К,introductory,0,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,train,train,104,False,False,False
3,116061850,4,Провадження № 1-кп/127/904/23\n\nВ И Р О К,ІМЕНЕМ УКРАЇНИ,introductory,0,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,train,train,104,False,False,False
4,116061850,5,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,28.12.2023 |місто Вінниця|,introductory,0,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,ВІННИЦЬКИЙ МІСЬКИЙ СУД ВІННИЦЬКОЇ ОБЛАСТІ,train,train,104,False,False,False
